# BRONZE_SIA

**Origem:** arquivos `PAAL{AA}{MM}.dbc` do SIA/SUS (produção ambulatorial de Alagoas), um por mês:
`ftp://ftp.datasus.gov.br/dissemin/publicos/SIASUS/200801_/Dados/`

Cada arquivo tem cerca de 600 mil linhas. A partição é o ano do arquivo (`ANO_MVM`,
coluna virtual derivada de `PA_MVM`).

**Idempotência:** a tabela é particionada por ano. Antes de carregar um ano, a partição dele é truncada (`TRUNCATE PARTITION FOR`), então reprocessar nunca duplica dados.

**Falhas:** se algum arquivo não puder ser baixado ou lido, o notebook para com erro e a task do Airflow fica vermelha.

Anos processados: variável de ambiente `ANOS_PIPELINE` (padrão `2023,2024`).

In [ ]:
import sys
sys.path.append("..")
from dw_utils import carregar_arquivos_mensais, consultar

In [ ]:
COLUNAS = ['PA_MVM', 'PA_CMP', 'PA_CODUNI', 'PA_GESTAO', 'PA_UFMUN', 'PA_TPUPS', 'PA_PROC_ID', 'PA_NIVCPL', 'PA_MUNPCN', 'PA_QTDPRO', 'PA_QTDAPR', 'PA_VALPRO', 'PA_VALAPR']

carregar_arquivos_mensais("BRONZE_SIA", "/dissemin/publicos/SIASUS/200801_/Dados", "PAAL", COLUNAS)

In [ ]:
consultar("""SELECT PA_MVM, COUNT(*) AS REGISTROS, SUM(PA_VALAPR) AS VALOR_APROVADO FROM BRONZE_SIA GROUP BY PA_MVM ORDER BY 1""")